Autograd is responsible for the backpropagation-algorithm. It searches the differentiation of every nod in the network starting from the very end.
Think of it like a stack beeing build on the way forward and accumulated on the way backwards.

In [1]:
import torch 

x = torch.ones(5)       # input tensor
y = torch.zeros(3)      # expected output

w = torch.randn(5, 3, requires_grad=True)   # weight-matrix
b = torch.randn(3, requires_grad=True)      # bias

z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [2]:
# tensors, functions and computonal graph
    # x, w --> matmul --> (+b) --> z --> BCE-Loss (through y) --> loss
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x75463123cb50>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x754631189c00>


In [3]:
# computing gradients
    # get the derivate of the loss function regarding to the parameters b and w
loss.backward()     # compute derivates

print(w.grad)
print(b.grad)

tensor([[0.0450, 0.1009, 0.3300],
        [0.0450, 0.1009, 0.3300],
        [0.0450, 0.1009, 0.3300],
        [0.0450, 0.1009, 0.3300],
        [0.0450, 0.1009, 0.3300]])
tensor([0.0450, 0.1009, 0.3300])


In [ ]:
# disable gradient tracking
    # when no training is happening or for fine tuning of an layer
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

# alternative way of disable tracking on one tensor
z = torch.matmul(x, w)+b
z_det = z.detach()      # disable tracking
print(z_det.requires_grad)

True
False
False


In [6]:
# tensor gradients and jacobian products
    # when the root is not a scalar loss we need a jacobian matrix of derivatives

inp = torch.eye(4, 5, requires_grad=True)   # diagonal is 1
out = (inp+1).pow(2).t()                    

out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n {inp.grad}")

# gradients get accumulated to the old ones
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")

inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n {inp.grad}")

First call
 tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
 tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
